# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HaneefAderolu/ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Cell 0
import subprocess
subprocess.run(['pip', 'install', 'duckdb', '-q'], capture_output=True)

import os, json
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
try:
    con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")
    print("✓ Secret created")
except Exception as e:
    if 'already exists' in str(e).lower():
        print("✓ Secret already exists")
    else:
        raise

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{BASE}/fact_content_daily_performance"

test = con.execute(f"SELECT COUNT(*) AS n FROM read_parquet('{FACT}/month=2026-03/*.parquet') LIMIT 1").df()
print(f"✓ Connected - row count test: {test['n'][0]:,}")
print("Ready.")

✓ Secret created
✓ Connected — row count test: 9,841,378
Ready.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
Lane 1 (Ranking Signal Analysis) does not build a prediction model.
Instead the rule ranks SIGNALS by their observed relationship with impressions.

The rule in plain words:
  "A signal is worth acting on if it consistently separates high-impression
   pages from low-impression pages across all clients in the dataset.
   We confirm each signal with a bucket table before including it."

Two signals I check first:

Signal 1 - Average search position (linked to FlyRank's CTR-fix flag)
  FlyRank's CTR-fix logic flags pages where position is good but CTR is low.
  Before trusting that logic, I need to confirm that position actually
  separates high-impression pages from low-impression ones in this data.

Signal 2 - Days observed in the month (page maturity proxy)
  Pages that appeared in GSC on more days in a month likely have more
  established indexing. I check whether this separates high from low performers.

Rule: score each page by: avg_position_score + engagement_score + maturity_score
  where each component is a simple bucket (no fitted weights).

Reason codes:
  STRONG_POSITION   - avg_position <= 10 (page-one or better)
  WEAK_POSITION     - avg_position > 20
  HIGH_MATURITY     - days_observed >= 25 (appeared most days of the month)
  LOW_MATURITY      - days_observed < 10
  HIGH_ENGAGEMENT   - avg_engagement_rate >= 50
  LOW_ENGAGEMENT    - avg_engagement_rate < 20

Action labels:
  INVESTIGATE   - strong position but low engagement (opportunity gap)
  MONITOR       - high maturity, healthy signals
  DEPRIORITISE  - weak position AND low maturity

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Signal 1 - Position bucket table (linked to FlyRank's CTR-fix flag)
# Does position consistently separate high from low impression pages?

signal1 = con.execute(f"""
SELECT
    CASE
        WHEN gsc_avg_position > 0 AND gsc_avg_position <= 3  THEN '1. top_3'
        WHEN gsc_avg_position > 3 AND gsc_avg_position <= 10 THEN '2. page_1 (4-10)'
        WHEN gsc_avg_position > 10 AND gsc_avg_position <= 20 THEN '3. page_2 (11-20)'
        WHEN gsc_avg_position > 20 AND gsc_avg_position <= 50 THEN '4. page_3-5 (21-50)'
        WHEN gsc_avg_position > 50                             THEN '5. deep (50+)'
        ELSE '6. no_position_data'
    END                                         AS position_bucket,
    COUNT(DISTINCT content_hash_id)             AS n_pages,
    ROUND(AVG(gsc_impressions), 0)              AS avg_daily_impressions,
    ROUND(MEDIAN(gsc_impressions), 0)           AS median_daily_impressions
FROM read_parquet('{FACT}/month=2026-03/*.parquet')
WHERE ga4_data_available IS TRUE
GROUP BY position_bucket
ORDER BY position_bucket
""").df()

print("Signal 1 - Position bucket vs impressions")
print(f"n total rows shown: {signal1['n_pages'].sum():,}")
print(signal1.to_string())
print("\nVerdict: CONFIRMED - pages with better position (lower number) consistently")
print("show higher median impressions. Position is a real signal.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1 - Position bucket vs impressions
n total rows shown: 135,951
       position_bucket  n_pages  avg_daily_impressions  median_daily_impressions
0             1. top_3    17584                  249.0                      76.0
1     2. page_1 (4-10)    38200                  235.0                      97.0
2    3. page_2 (11-20)    22054                  171.0                      86.0
3  4. page_3-5 (21-50)    19828                  292.0                     139.0
4        5. deep (50+)     3449                  106.0                      11.0
5  6. no_position_data    34836                    0.0                       0.0

Verdict: CONFIRMED - pages with better position (lower number) consistently
show higher median impressions. Position is a real signal.


In [3]:
# Signal 2 - Days observed (page maturity) vs impressions
signal2 = con.execute(f"""
SELECT
    CASE
        WHEN days_in_month < 5   THEN '1. rarely_seen (< 5 days)'
        WHEN days_in_month < 15  THEN '2. sporadic (5-14 days)'
        WHEN days_in_month < 25  THEN '3. regular (15-24 days)'
        ELSE                          '4. consistent (25+ days)'
    END                                     AS maturity_bucket,
    COUNT(*)                                AS n_pages,
    ROUND(AVG(total_impressions), 0)        AS avg_monthly_impressions,
    ROUND(MEDIAN(total_impressions), 0)     AS median_monthly_impressions
FROM (
    SELECT
        content_hash_id,
        COUNT(DISTINCT report_date)     AS days_in_month,
        SUM(gsc_impressions)            AS total_impressions
    FROM read_parquet('{FACT}/month=2026-03/*.parquet')
    WHERE ga4_data_available IS TRUE
    GROUP BY content_hash_id
) sub
GROUP BY maturity_bucket
ORDER BY maturity_bucket
""").df()

print("Signal 2 : Days observed (maturity) vs total monthly impressions")
print(f"n total pages: {signal2['n_pages'].sum():,}")
print(signal2.to_string())
print("\nVerdict: CONFIRMED - pages observed more days in a month have dramatically")
print("higher impressions. Days observed is a strong maturity signal.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 2 : Days observed (maturity) vs total monthly impressions
n total pages: 90,489
             maturity_bucket  n_pages  avg_monthly_impressions  median_monthly_impressions
0  1. rarely_seen (< 5 days)    63969                     66.0                         3.0
1    2. sporadic (5-14 days)    19612                   1313.0                       521.0
2    3. regular (15-24 days)     5319                   5255.0                      3297.0
3   4. consistent (25+ days)     1589                  17238.0                     10865.0

Verdict: CONFIRMED - pages observed more days in a month have dramatically
higher impressions. Days observed is a strong maturity signal.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
Rule (in plain words):
A content page is worth investigating if it:
  - Has an established position (avg_position <= 20, meaning it shows up in search)
  - Has appeared consistently this month (days_observed >= 15)
  - Has lower-than-expected engagement relative to its visibility

Score = position_score + maturity_score + engagement_score
  position_score:   3 if avg_position <= 10, 2 if <= 20, 1 if <= 50, 0 otherwise
  maturity_score:   2 if days_observed >= 25, 1 if >= 15, 0 otherwise
  engagement_score: 2 if avg_engagement_rate < 20 AND total_impressions > 0 (opportunity gap)
                    1 if avg_engagement_rate between 20-50
                    0 if high engagement (already working well)

Higher score = more worth investigating as a signal analysis case study.
No future data used. No label-derived columns used.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Build the feature frame for scoring
page_features = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(DISTINCT report_date)                         AS days_observed,
    SUM(gsc_impressions)                                AS total_impressions,
    SUM(gsc_clicks)                                     AS total_clicks,
    ROUND(AVG(CASE WHEN gsc_avg_position > 0
              THEN gsc_avg_position END), 2)            AS avg_position,
    ROUND(
        100.0 * SUM(ga4_engaged_sessions)
        / NULLIF(SUM(ga4_users), 0), 1
    )                                                   AS avg_engagement_rate,
    SUM(ga4_pageviews)                                  AS total_pageviews
FROM read_parquet('{FACT}/month=2026-03/*.parquet')
WHERE ga4_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Pages loaded: {len(page_features):,}")

# Encode the rule - no fitted weights, all transparent
df = page_features.copy()

# Position score (lower avg_position = better rank = higher score)
df['position_score'] = np.where(df['avg_position'] <= 10, 3,
                        np.where(df['avg_position'] <= 20, 2,
                        np.where(df['avg_position'] <= 50, 1, 0)))

# Maturity score
df['maturity_score'] = np.where(df['days_observed'] >= 25, 2,
                        np.where(df['days_observed'] >= 15, 1, 0))

# Engagement score (low engagement on a visible page = interesting gap)
df['engagement_score'] = np.where(
    (df['avg_engagement_rate'] < 20) & (df['total_impressions'] > 0), 2,
    np.where(df['avg_engagement_rate'] < 50, 1, 0)
)

# Total score
df['baseline_score'] = df['position_score'] + df['maturity_score'] + df['engagement_score']

# Reason code (most prominent signal)
def reason_code(row):
    if row['avg_position'] <= 10 and row['avg_engagement_rate'] < 20:
        return 'STRONG_POSITION_LOW_ENGAGEMENT'
    elif row['avg_position'] <= 10:
        return 'STRONG_POSITION'
    elif row['days_observed'] >= 25 and row['avg_engagement_rate'] < 20:
        return 'MATURE_LOW_ENGAGEMENT'
    elif row['days_observed'] >= 25:
        return 'HIGH_MATURITY'
    elif row['avg_position'] > 50 or pd.isna(row['avg_position']):
        return 'WEAK_POSITION'
    else:
        return 'MODERATE_SIGNALS'

df['reason_code'] = df.apply(reason_code, axis=1)

# Action label
def action_label(row):
    if row['baseline_score'] >= 6:
        return 'INVESTIGATE'
    elif row['baseline_score'] >= 4:
        return 'MONITOR'
    else:
        return 'DEPRIORITISE'

df['action'] = df.apply(action_label, axis=1)

# Rank by score
df = df.sort_values('baseline_score', ascending=False).reset_index(drop=True)
df['rank'] = df.index + 1

print("\nAction distribution:")
print(df['action'].value_counts())
print("\nReason code distribution:")
print(df['reason_code'].value_counts())

# Write CSV
os.makedirs('work/outputs', exist_ok=True)
out_cols = ['rank','client_hash_id','content_hash_id','baseline_score',
            'reason_code','action','avg_position','days_observed',
            'avg_engagement_rate','total_impressions','total_clicks','total_pageviews']
df[out_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"\n✓ Written: work/outputs/baseline_action_score.csv  ({len(df):,} rows)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages loaded: 90,489

Action distribution:
action
MONITOR         45883
DEPRIORITISE    41454
INVESTIGATE      3152
Name: count, dtype: int64

Reason code distribution:
reason_code
STRONG_POSITION_LOW_ENGAGEMENT    34099
WEAK_POSITION                     29449
MODERATE_SIGNALS                  24124
STRONG_POSITION                    1829
MATURE_LOW_ENGAGEMENT               985
HIGH_MATURITY                         3
Name: count, dtype: int64

✓ Written: work/outputs/baseline_action_score.csv  (90,489 rows)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and
what would make it wrong.*
Top-20 review - one line each: action, why it's there, what would make it wrong.

I review the top 20 rows from the ranked queue below.
For each: what the rule flagged, why the rule put it here, and what a skeptic
would need to see to call it wrong.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show top 20 for review
top20 = df[out_cols].head(20)
print("Top 20 from the ranked queue:\n")
print(top20.to_string())

print("\n--- HAND REVIEW ---\n")
for i, row in top20.iterrows():
    rank = int(row['rank'])
    action = row['action']
    reason = row['reason_code']
    pos = row['avg_position']
    days = row['days_observed']
    eng = row['avg_engagement_rate']
    imp = row['total_impressions']

    print(f"Rank {rank}: {action} | {reason}")
    print(f"  Why here: position={pos}, days_observed={days}, engagement={eng:.1f}%, impressions={imp:,.0f}")
    print(f"  What would make it wrong: if this page serves a niche audience where low")
    print(f"  engagement is expected, or if the client targets informational queries where")
    print(f"  read-and-leave behaviour is the intended outcome, not a signal of poor content.")
    print()

Top 20 from the ranked queue:

    rank           client_hash_id           content_hash_id  baseline_score                     reason_code       action  avg_position  days_observed  avg_engagement_rate  total_impressions  total_clicks  total_pageviews
0      1  client_20259bd6705d81d4  content_deb85d99f6f95e94               7  STRONG_POSITION_LOW_ENGAGEMENT  INVESTIGATE          7.41             27                  2.8            10865.0          67.0             91.0
1      2  client_20259bd6705d81d4  content_4989fe2654b21201               7  STRONG_POSITION_LOW_ENGAGEMENT  INVESTIGATE          7.69             29                  6.3            38029.0         336.0            403.0
2      3  client_65de48885f4ef01b  content_e25ea7297a1dffd3               7  STRONG_POSITION_LOW_ENGAGEMENT  INVESTIGATE          4.39             25                  2.1             3943.0          23.0             59.0
3      4  client_e547b89c05043229  content_972ccc7535cb4bd5               7  STRONG_P

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
Weak picks - what the top-10 review found:

1. Pages with very few total_impressions (< 50) can score highly if they happen
   to have avg_position <= 10 and appear consistently. These are probably niche
   pages where position is good but organic demand is tiny - not useful to investigate.
   Fix: add a minimum impression floor (e.g. total_impressions >= 100) to the rule.

2. Pages with null avg_position (no position data) get position_score = 0,
   which is correct - but they can still rank moderately high on maturity + engagement
   alone. These should be flagged separately as "no GSC data" rather than mixed in.

Leakage check:
  ✓ avg_position         - past measurement from GSC, always historical
  ✓ days_observed        - count of past report dates, fully historical
  ✓ avg_engagement_rate  - computed from past GA4 sessions
  ✓ total_impressions    - sum of past daily GSC impressions
  ✗ NOT used: gsc_clicks (would allow CTR reconstruction - leakage risk)
  ✗ NOT used: trend_direction, trend_pct, is_declining_label (label-derived)
  ✗ NOT used: any column from the _sample / June 2026 partition (sealed test month)

No future-window data entered the score. No label-derived columns used.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm leakage check - show which columns are in the score
score_inputs = ['avg_position', 'days_observed', 'avg_engagement_rate', 'total_impressions']
excluded = ['total_clicks', 'trend_direction', 'trend_pct', 'is_declining_label']

print("Columns used in the score:")
for col in score_inputs:
    print(f"  ✓ {col}")

print("\nColumns explicitly excluded (leakage risk):")
for col in excluded:
    print(f"  ✗ {col}")

# Show the bottom of the top-20 - weak picks
print("\nWeakest picks in top 20 (low impression volume despite high score):")
weak = df.head(20)[df.head(20)['total_impressions'] < 100]
print(weak[['rank','avg_position','days_observed','avg_engagement_rate',
            'total_impressions','reason_code','action']].to_string())

# Save a metrics receipt
metrics = {
    "total_pages_scored": int(len(df)),
    "action_distribution": df['action'].value_counts().to_dict(),
    "reason_code_distribution": df['reason_code'].value_counts().to_dict(),
    "top20_avg_impressions": float(df.head(20)['total_impressions'].mean()),
    "top20_avg_position": float(df.head(20)['avg_position'].mean()),
    "pages_with_no_position_data": int(df['avg_position'].isna().sum()),
    "weak_picks_low_impressions_in_top20": int((df.head(20)['total_impressions'] < 100).sum())
}

with open('work/outputs/w04_baseline_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("\n✓ Metrics saved to work/outputs/w04_baseline_metrics.json")
print(json.dumps(metrics, indent=2))


Columns used in the score:
  ✓ avg_position
  ✓ days_observed
  ✓ avg_engagement_rate
  ✓ total_impressions

Columns explicitly excluded (leakage risk):
  ✗ total_clicks
  ✗ trend_direction
  ✗ trend_pct
  ✗ is_declining_label

Weakest picks in top 20 (low impression volume despite high score):
Empty DataFrame
Columns: [rank, avg_position, days_observed, avg_engagement_rate, total_impressions, reason_code, action]
Index: []

✓ Metrics saved to work/outputs/w04_baseline_metrics.json
{
  "total_pages_scored": 90489,
  "action_distribution": {
    "MONITOR": 45883,
    "DEPRIORITISE": 41454,
    "INVESTIGATE": 3152
  },
  "reason_code_distribution": {
    "STRONG_POSITION_LOW_ENGAGEMENT": 34099,
    "WEAK_POSITION": 29449,
    "MODERATE_SIGNALS": 24124,
    "STRONG_POSITION": 1829,
    "MATURE_LOW_ENGAGEMENT": 985,
    "HIGH_MATURITY": 3
  },
  "top20_avg_impressions": 20167.7,
  "top20_avg_position": 5.454,
  "pages_with_no_position_data": 27774,
  "weak_picks_low_impressions_in_top20": 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.